# Voxae bridge training
Thin wrapper around `voxae.train.train`. Runtime: **L4 or A100** (the configs train in bfloat16).

```bash
uv run python scripts/package_for_colab.py --max-px 1536
```

Then upload `data/colab_bundle.zip` to Drive.

In [ ]:
!nvidia-smi
# Absolute paths keep the cell re-runnable after a partial run.
%cd /content
![ -d voxae ] || git clone https://github.com/nhipixel/voxae.git
%cd /content/voxae
!git pull --ff-only
!pip install -q -e ".[ml,data]" bitsandbytes peft wandb
# peft raises on Colab's preinstalled torchao; quantization goes through bitsandbytes.
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Unzip to local disk: Drive is slow for many small reads during training.
!cp /content/drive/MyDrive/voxae/colab_bundle.zip /content/
!unzip -q /content/colab_bundle.zip -d /content/bundle
!ls /content/bundle && wc -l /content/bundle/processed/annotations/*.jsonl

In [ ]:
# Rewrite the configs for this VM. Output goes to a .colab.yaml copy so the
# checkout stays clean and `git pull` keeps working.
import pathlib

import yaml

BUNDLE = "/content/bundle"
JSONL = BUNDLE + "/processed/annotations/voxae_reason.jsonl"
# Reports go here, outside both the bundle and the checkpoints tree. Written
# inside the bundle they die with the VM, which is how the first test-split
# reports were lost; written beside the checkpoints they would ride along with
# several gigabytes of ablation weights every time those get copied to Drive.
RESULTS = "/content/results"

configs = [pathlib.Path(f"voxae/train/configs/{n}.yaml") for n in ("smoke_2b", "full_2b")]
configs += sorted(pathlib.Path("voxae/train/configs/ablations").glob("*.yaml"))
for p in configs:
    cfg = yaml.safe_load(p.read_text())
    cfg["data_root"] = BUNDLE
    cfg["train_jsonl"] = JSONL
    cfg["output_dir"] = "/content/" + cfg["output_dir"]
    p.with_suffix(".colab.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
    print(f"{p.stem:20s} -> {cfg['output_dir']}")

## Smoke test
100 samples. `loss_mask` falling toward zero is the signal that the bridge learns.

In [ ]:
!python -m voxae.train.train --config voxae/train/configs/smoke_2b.colab.yaml

## Full run
All training samples. Checkpoints resume automatically, so a disconnect costs only the steps since the last save.

In [ ]:
import os

os.environ["WANDB_API_KEY"] = ""  # optional
!python -m voxae.train.train --config voxae/train/configs/full_2b.colab.yaml

In [ ]:
# Persist checkpoints before the session ends.
!mkdir -p /content/drive/MyDrive/voxae/outputs
!cp -r /content/outputs/* /content/drive/MyDrive/voxae/outputs/
!du -sh /content/drive/MyDrive/voxae/outputs/*

In [ ]:
# Quick look at the loss curve.
import json
import pathlib

import matplotlib.pyplot as plt

log = pathlib.Path("/content/outputs/full_2b/train_log.jsonl")
recs = [json.loads(line) for line in log.read_text().splitlines() if line.strip()]
for key in ("loss", "loss_mask", "loss_ce"):
    if key in recs[0]:
        plt.plot([r["step"] for r in recs], [r[key] for r in recs], label=key)
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

## Evaluation
Both predictors score on the same split, giving gIoU/cIoU overall and per query family. The baseline calls a hosted VLM and needs an API key.

In [ ]:
!python -m voxae.eval.run_eval \
    --split test --predictor trained \
    --checkpoint /content/outputs/full_2b/latest \
    --backbone Qwen/Qwen2-VL-2B-Instruct \
    --data-root /content/bundle --device cuda \
    --out-dir {RESULTS}

In [ ]:
import os

os.environ["VOXAE_VLM_API_KEY"] = ""  # required for the baseline only
!python -m voxae.eval.run_eval \
    --split test --predictor zero-shot \
    --data-root /content/bundle \
    --out-dir {RESULTS}

In [ ]:
# Persist the reports the moment they exist, before anything else can go wrong.
# The absence of this step is what lost the first test-split reports: eval wrote
# them inside the bundle and only the checkpoints were copied back.
!mkdir -p /content/drive/MyDrive/voxae/results
!cp -r {RESULTS}/. /content/drive/MyDrive/voxae/results/
!ls -la /content/drive/MyDrive/voxae/results/

In [ ]:
# The published tables, printed from the reports rather than typed from them.
!python scripts/results_table.py --results-dir {RESULTS} \
    --check {JSONL} --out {RESULTS}/results.md

## Ablations

Five arms sharing one step budget, one seed, and one eval split. The control is the shipped recipe at the same reduced budget, so the arms compare to each other even though every number sits below the full run's.

Evaluation is on **val**, not test. Choosing anything on the test split would contaminate the headline number.

Each arm owns its output directory and resume is keyed on step count, so re-running this cell after a disconnect continues the unfinished arm and skips the finished ones. Evaluation resumes from its own records the same way, so a completed arm costs nothing on a second pass.

In [ ]:
# One arm at a time: train, score on val, persist. subprocess rather than shell
# magic so a failed arm stops the grid instead of letting the next eval score a
# checkpoint that was never written. No --fresh, so a rerun after a disconnect
# resumes the unfinished arm and the finished ones exit at once.
import shutil
import subprocess

ARMS = ["base", "no_ce", "dice_light", "projector_1layer", "sam_frozen"]
DRIVE_RESULTS = "/content/drive/MyDrive/voxae/results"

for arm in ARMS:
    print(f"===== {arm} =====", flush=True)
    subprocess.run(
        [
            "python", "-m", "voxae.train.train",
            "--config", f"voxae/train/configs/ablations/{arm}.colab.yaml",
        ],
        check=True,
    )
    subprocess.run(
        [
            "python", "-m", "voxae.eval.run_eval",
            "--split", "val", "--predictor", "trained",
            "--checkpoint", f"/content/outputs/ablations/{arm}/latest",
            "--backbone", "Qwen/Qwen2-VL-2B-Instruct",
            "--data-root", BUNDLE, "--device", "cuda",
            "--out-dir", RESULTS, "--tag", arm,
        ],
        check=True,
    )
    shutil.copytree(RESULTS, DRIVE_RESULTS, dirs_exist_ok=True)
    print(f"{arm} scored and persisted", flush=True)


In [ ]:
# The ablation table, alongside the headline one.
!python scripts/results_table.py --results-dir {RESULTS} --control base     --check {JSONL} --out {RESULTS}/results.md